# RAG on Orders — Semantic Search Over Support Notes

In [ ]:
from google.colab import auth, files
auth.authenticate_user()

!pip install --quiet google-cloud-bigquery

from google.cloud import bigquery
import pandas as pd, time, json

PROJECT_ID = "project-2df9d2ad-b6f5-4ed5-997"
DATASET = "orders"
ORDERS_TABLE = "orders_sample"
LOCATION = "US"

client = bigquery.Client(project=PROJECT_ID)

def run(sql):
    return client.query(sql).result().to_dataframe()


## Load synthetic support notes
One free-text note per order -- the piece orders_raw never had.

In [ ]:
uploaded = files.upload()  # choose order_notes.csv
notes_df = pd.read_csv("order_notes.csv")

job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
client.load_table_from_dataframe(notes_df, f"{PROJECT_ID}.{DATASET}.order_notes", job_config=job_config).result()
print("loaded", len(notes_df), "notes")


## Connect to an embedding model

In [ ]:
CONNECTION_ID = f"{PROJECT_ID}.{LOCATION}.embedding_conn"
!bq mk --connection --connection_type=CLOUD_RESOURCE --project_id={PROJECT_ID} --location={LOCATION} embedding_conn

raw = !bq show --format=prettyjson --connection {CONNECTION_ID}
embedding_sa = json.loads("\n".join(raw))["cloudResource"]["serviceAccountId"]
print("service account:", embedding_sa)

time.sleep(45)
!gcloud projects add-iam-policy-binding {PROJECT_ID} --member="serviceAccount:{embedding_sa}" --role="roles/owner" --quiet --condition=None
time.sleep(60)

run(f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.{DATASET}.embedding_model`
REMOTE WITH CONNECTION `{CONNECTION_ID}`
OPTIONS (endpoint = 'text-embedding-005')
""")


## Embed the notes

In [ ]:
run(f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET}.note_embeddings` AS
SELECT order_id, note, ml_generate_embedding_result AS embedding
FROM ML.GENERATE_EMBEDDING(
  MODEL `{PROJECT_ID}.{DATASET}.embedding_model`,
  (SELECT order_id, note, note AS content FROM `{PROJECT_ID}.{DATASET}.order_notes`)
)
""")
print("embeddings ready")


## Ask a question in plain language
Finds semantically similar past notes, then joins back to the real order.

In [ ]:
QUESTION = "customer says the package looked tampered with"

display(run(f"""
SELECT vs.base.order_id AS order_id, vs.base.note AS note, vs.distance,
       o.country, o.category, o.amount, o.order_date
FROM VECTOR_SEARCH(
  TABLE `{PROJECT_ID}.{DATASET}.note_embeddings`, 'embedding',
  (SELECT ml_generate_embedding_result AS embedding
   FROM ML.GENERATE_EMBEDDING(MODEL `{PROJECT_ID}.{DATASET}.embedding_model`,
     (SELECT '{QUESTION}' AS content))),
  top_k => 5
) AS vs
JOIN `{PROJECT_ID}.{DATASET}.{ORDERS_TABLE}` o ON vs.base.order_id = o.order_id
ORDER BY vs.distance
"""))

Try a different question:

In [ ]:
QUESTION = "the item stopped working after a couple of days"

display(run(f"""
SELECT vs.base.order_id AS order_id, vs.base.note AS note, vs.distance,
       o.country, o.category, o.amount, o.order_date
FROM VECTOR_SEARCH(
  TABLE `{PROJECT_ID}.{DATASET}.note_embeddings`, 'embedding',
  (SELECT ml_generate_embedding_result AS embedding
   FROM ML.GENERATE_EMBEDDING(MODEL `{PROJECT_ID}.{DATASET}.embedding_model`,
     (SELECT '{QUESTION}' AS content))),
  top_k => 5
) AS vs
JOIN `{PROJECT_ID}.{DATASET}.{ORDERS_TABLE}` o ON vs.base.order_id = o.order_id
ORDER BY vs.distance
"""))

In [ ]:
display(run(f"""
SELECT
  order_id, note,
  AI.GENERATE(
    prompt => CONCAT(
      'Classify the sentiment of this customer support note as exactly one word: ',
      'Positive, Negative, or Neutral. Note: ', note
    ),
    connection_id => '{PROJECT_ID}.{LOCATION}.embedding_conn',
    output_schema => 'sentiment STRING'
  ).sentiment AS sentiment,
  AI.GENERATE(
    prompt => CONCAT(
      'Rate how severe this customer support issue is on a scale of 1 (minor) to 5 (urgent). Note: ', note
    ),
    connection_id => '{PROJECT_ID}.{LOCATION}.embedding_conn',
    output_schema => 'severity INT64'
  ).severity AS severity
FROM `{PROJECT_ID}.{DATASET}.order_notes`
ORDER BY severity DESC
"""))

## Cleanup

In [ ]:
# Cleanup -- uncomment to drop what this notebook created
run(f"DROP TABLE `{PROJECT_ID}.{DATASET}.order_notes`")
run(f"DROP TABLE `{PROJECT_ID}.{DATASET}.note_embeddings`")
run(f"DROP MODEL `{PROJECT_ID}.{DATASET}.embedding_model`")
!bq rm --connection --project_id={PROJECT_ID} --location={LOCATION} -f embedding_conn
